In [42]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools

In [43]:
# Suma kontrolna (hash) danych w pliku Parquet, z uwzględnieniem sortowania po kolumnach TowId i Data.

nazwa_pliku = "fact_inka_soft.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   fact_inka_soft.parquet   87ff0e55ac94368b71e9e36a33c43e321c2bfaeeaffae189daa9d0d8996cff10


In [44]:
df_kalendarz = pd.read_parquet(f"dane/interim/{nazwa_pliku}")


In [45]:


df_sprzedaz = df_kalendarz[df_kalendarz['TypDok'] == 21].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")


TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161


In [46]:

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)


# Zakres życia per TowId (z bufor 14 dni,)
daty_sku = df_sprzedaz.groupby('TowId')['Data'].agg(DataStart='min', DataKoniec='max').reset_index()
daty_sku['DataKoniec'] = (daty_sku['DataKoniec'] + pd.Timedelta(days=14)).clip(upper=data_max)

# Dołączamy do pelny_kalendarz i filtrujemy
pelny_kalendarz = pelny_kalendarz.merge(daty_sku, on='TowId', how='left')

przed = len(pelny_kalendarz)
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['DataStart']) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['DataKoniec'])
].copy()

print(f"Przed przycięciem: {przed:,} wierszy")
print(f"Po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# Opcjonalnie: usuń pomocnicze kolumny DataStart/DataKoniec, jeśli nie chcesz ich w finalnym pliku
pelny_kalendarz = pelny_kalendarz.drop(columns=['DataStart', 'DataKoniec'])




print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


Przed przycięciem: 15,810,161 wierszy
Po przycięciu:     8,802,515 wierszy

Puste rekordy (DokId=-1): 5,428,540
Realne rekordy sprzedaży: 3,373,975


In [47]:
# Uzupełnianie brakujące wartości w kolumnach
pelny_kalendarz['Kolejnosc'] = pelny_kalendarz['Kolejnosc'].fillna(0)
pelny_kalendarz['NrPozycji'] = pelny_kalendarz['NrPozycji'].fillna(0)
pelny_kalendarz['TypPoz'] = pelny_kalendarz['TypPoz'].fillna(4)
pelny_kalendarz['IloscPlus'] = pelny_kalendarz['IloscPlus'].fillna(0)
pelny_kalendarz['IloscMinus'] = pelny_kalendarz['IloscMinus'].fillna(0)
pelny_kalendarz['CenaPoRab'] = pelny_kalendarz['CenaPoRab'].ffill()
pelny_kalendarz['Wartosc'] = pelny_kalendarz['Wartosc'].fillna(0)
pelny_kalendarz['KolejnyWDniu'] = pelny_kalendarz['KolejnyWDniu'].fillna(-1)
pelny_kalendarz['NrDok'] = pelny_kalendarz['NrDok'].fillna('brak')
pelny_kalendarz['TypDok'] = pelny_kalendarz['TypDok'].fillna(21)
pelny_kalendarz['AktywnyDok'] = pelny_kalendarz['AktywnyDok'].fillna(1)
pelny_kalendarz['Razem'] = pelny_kalendarz['Razem'].fillna(0)
pelny_kalendarz['DoZaplaty'] = pelny_kalendarz['DoZaplaty'].fillna(0)
pelny_kalendarz['Zaplacono'] = pelny_kalendarz['Zaplacono'].fillna(0)
pelny_kalendarz['Opis1'] = pelny_kalendarz['Opis1'].fillna('brak')
pelny_kalendarz['Producent'] = pelny_kalendarz['Producent'].fillna(-1)
pelny_kalendarz['AktywnyTow'] = pelny_kalendarz['AktywnyTow'].fillna(1)
pelny_kalendarz['Dokument'] = pelny_kalendarz['Dokument'].fillna("DF")
pelny_kalendarz['WplywNaStan'] = pelny_kalendarz['WplywNaStan'].fillna(False).astype(int)
pelny_kalendarz['MetodaLiczenia'] = pelny_kalendarz['MetodaLiczenia'].fillna('IP')
pelny_kalendarz['Mnoznik'] = pelny_kalendarz['Mnoznik'].fillna(0)
pelny_kalendarz['TypRuchu'] = pelny_kalendarz['TypRuchu'].fillna('sprzedaz')
pelny_kalendarz['CzyNiechciane'] = pelny_kalendarz['CzyNiechciane'].fillna(False).astype(int)


In [48]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


Wiersze w df_sprzedaz (surowe): 3,373,975
Realne rekordy w pelny_kalendarz: 3,373,975
Zgodność: True

Rozmiar: (8802515, 30)
Pamięć: 2760.3 MB

Kolumny: ['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow', 'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane']


In [49]:
pelny_kalendarz.to_parquet(
    "dane/interim/fact_inka_calendar.parquet",
    compression='zstd',
    index=False
)


In [50]:
# Suma kontrolna
nazwa_pliku = "fact_inka_calendar.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   fact_inka_calendar.parquet   a36b20e62c8328186b8c2be3457e4c43a51d08664ebf9864086d2cb502ae5d57


In [ ]:
# hash pliku kalendarz_pelny_towid.parquet:  a36b20e62c8328186b8c2be3457e4c43a51d08664ebf9864086d2cb502ae5d57